# Deliverable 3 — Supervised modeling

Predicts **`success_composite_v1`** from **22** features: **`nba_debut_age`**, four rookie NBA position dummies (`pos_is_SG`, `pos_is_SF`, `pos_is_PF`, `pos_is_C`), and **17 college advanced** columns (recruiting excluded). College `cbb_*` on `model_base` are each player's **last NCAA season** (not the SR **Career** row). **`Pos`** is taken from the **earliest** deduped NBA season row; only **PG / SG / SF / PF / C** are used (first token if hyphenated). **PG is the reference** (all four dummies are 0 for point guards); missing or odd labels use that same baseline (no `pos_is_UNK` in this cohort).

Same logic as **`python -m src.models.evaluate_supervised`**:

- **Complete case:** drop rows with missing **`cbb_advanced_BPM`**.
- **Pipelines:** `SimpleImputer(median)` + model (no fit on full data before split).
- **Tune** with **5-fold CV** on training only; **evaluate** on held-out **20%** test.
- **Models:** Ridge (L2 baseline), Lasso via `LassoCV` (L1 feature selection), random forest, gradient boosting (`HistGradientBoostingRegressor`).
- **Feature selection:** Lasso zeros out uninformative coefficients; see `lasso_cv.selected_features` in the JSON for features retained.
- **Explain:** permutation importance uses the best test-R² among **ridge / random forest / gradient boosting** (Lasso excluded when sparse—see `03_supervised.md` §4). Table: `outputs/supervised/permutation_importance.csv`. JSON includes **`best_test_r2_model`** (all four models).

**Where the code lives:** Pipelines, CV, metrics, and permutation importance are implemented in **`src/models/evaluate_supervised.py`**. The modeling table and feature list are built in **`src/models/training_data.py`**. This notebook calls `evaluate_supervised.main()` so results match the CLI.

**Reading outputs:** The next cell prints a long JSON blob; some UIs truncate the middle. Use the **following cell** for test RMSE / MAE / R², or inspect `results["ridge"]["test"]["r2"]` in code. **`progress/03_supervised.md`** is written separately; update it after substantive reruns if you want the report to match fresh numbers (the script always refreshes the CSV and figure).

In [1]:
from pathlib import Path
import importlib
import sys

_cands = [Path(".").resolve(), Path("..").resolve(), Path.cwd()]
ROOT = next(p for p in _cands if (p / "data" / "processed" / "model_base_player_season.csv").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

evaluate_supervised = importlib.import_module("src.models.evaluate_supervised")
results = evaluate_supervised.main()

{
  "n_total": 595,
  "n_train": 476,
  "n_test": 119,
  "n_features": 22,
  "feature_names": [
    "nba_debut_age",
    "pos_is_SG",
    "pos_is_SF",
    "pos_is_PF",
    "pos_is_C",
    "cbb_advanced_PER",
    "cbb_advanced_BPM",
    "cbb_advanced_OBPM",
    "cbb_advanced_DBPM",
    "cbb_advanced_WS_40",
    "cbb_advanced_USGpct",
    "cbb_advanced_TSpct",
    "cbb_advanced_ASTpct",
    "cbb_advanced_TRBpct",
    "cbb_advanced_ORBpct",
    "cbb_advanced_DRBpct",
    "cbb_advanced_STLpct",
    "cbb_advanced_BLKpct",
    "cbb_advanced_TOVpct",
    "cbb_advanced_OWS",
    "cbb_advanced_DWS",
    "cbb_advanced_WS"
  ],
  "ridge": {
    "best_params": {
      "model__alpha": 81.8546730706903
    },
    "cv_best_rmse": 0.8277615682653627,
    "test": {
      "rmse": 0.8925388748595353,
      "mae": 0.733765130057003,
      "r2": 0.13940557683344057
    }
  },
  "lasso_cv": {
    "alpha": 0.044704665472750996,
    "cv_best_rmse": 0.8246806023493326,
    "n_nonzero_coefs": 5,
    "selected_f

In [2]:
import pandas as pd
from IPython.display import display

display(
    pd.DataFrame(
        {
            "ridge": results["ridge"]["test"],
            "lasso_cv": results["lasso_cv"]["test"],
            "random_forest": results["random_forest"]["test"],
            "gradient_boosting": results["gradient_boosting"]["test"],
        }
    )
)
lasso = results["lasso_cv"]
print(f"Lasso alpha: {lasso['alpha']:.4f}  |  nonzero coefficients: {lasso['n_nonzero_coefs']} / {results['n_features']}")
print(f"Lasso selected features: {lasso['selected_features']}")
print(f"\nBest test R² (all models): {results['best_test_r2_model']}")
print(f"Permutation importance model (ridge/RF/GB only): {results['permutation_importance_model']}")
print("Top permutation importances:", results["permutation_importance_top10"])

,ridge,lasso_cv,random_forest,gradient_boosting
rmse,0.892539,0.885633,0.901475,0.900311
mae,0.733765,0.725471,0.733743,0.735601
r2,0.139406,0.152672,0.122087,0.124353


Lasso alpha: 0.0447  |  nonzero coefficients: 5 / 22
Lasso selected features: ['nba_debut_age', 'cbb_advanced_BPM', 'cbb_advanced_DBPM', 'cbb_advanced_STLpct', 'cbb_advanced_TOVpct']

Best test R² (all models): lasso_cv
Permutation importance model (ridge/RF/GB only): ridge
Top permutation importances: {'nba_debut_age': 0.19707, 'cbb_advanced_BPM': 0.03275, 'cbb_advanced_OBPM': 0.01981, 'cbb_advanced_TSpct': 0.01154, 'cbb_advanced_ASTpct': 0.00975, 'cbb_advanced_USGpct': 0.00753, 'pos_is_PF': 0.00611, 'cbb_advanced_PER': 0.00403, 'cbb_advanced_DBPM': 0.003, 'pos_is_SF': 0.00297}
